# 京东方A (000725.SZ) 收盘价走势分析

> 数据来源：腾讯自选股行情接口 · 时间：2025-07-02 ~ 2026-07-02

本 Notebook 演示：
1. 从本地 JSON 加载日线数据
2. 数据清洗与探索
3. 绘制收盘价曲线 + 移动均线（MA5/MA20/MA60）


## 1. 环境准备


In [ ]:
!pip install pandas matplotlib numpy -q


## 2. 导入依赖 & 中文字体配置


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# ========================================
# 中文字体配置（按优先级尝试）
# ========================================
plt.rcParams['font.sans-serif'] = [
    'Arial Unicode MS',
    'PingFang SC',
    'Heiti SC',
    'SimHei',
    'Microsoft YaHei',
    'WenQuanYi Micro Hei'
]
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

print("环境就绪 ✅")


## 3. 加载数据

从本地 JSON 文件读取京东方A的日线数据（开盘价、收盘价、最高价、最低价、成交量等）。


In [ ]:
# ========================================
# 从本地 JSON 加载数据
# ========================================
with open("boe_kline_data.json", "r") as f:
    raw_data = json.load(f)

print(f"✅ 加载完成，共 {len(raw_data)} 条日线记录")


## 4. 数据预览

查看原始 JSON 的前几条记录，了解数据结构。


In [ ]:
# 查看前 3 条原始记录
for i, record in enumerate(raw_data[:3]):
    print(f"--- 记录 {i+1} ---")
    for k, v in record.items():
        print(f"  {k}: {v}")
    print()

print(f"... 共 {len(raw_data)} 条记录")


## 5. 转换为 DataFrame 并清洗


In [ ]:
# ========================================
# 转换为 DataFrame
# ========================================
df = pd.DataFrame(raw_data)

# 日期列转 datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 计算涨跌幅
df['change_pct'] = ((df['close'] - df['close'].shift(1)) / df['close'].shift(1) * 100).round(2)

# 添加涨跌标记
df['up'] = df['close'] >= df['open']

print(f"数据维度: {df.shape}")
print(f"日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"交易日数: {len(df)}")
print()
print("前 5 行预览：")
df.head()


## 6. 统计摘要


In [ ]:
# ========================================
# 基本统计
# ========================================
first_close = df['close'].iloc[0]
last_close  = df['close'].iloc[-1]
total_return = (last_close / first_close - 1) * 100

print("=" * 55)
print("    京东方A (000725.SZ) 收盘价统计")
print("=" * 55)
print(f"  首日收盘价:    ¥{first_close:.2f}")
print(f"  最新收盘价:    ¥{last_close:.2f}")
print(f"  累计涨跌幅:    {total_return:+.2f}%")
print(f"  最高收盘价:    ¥{df['close'].max():.2f}  ({df.loc[df['close'].idxmax(), 'date'].date()})")
print(f"  最低收盘价:    ¥{df['close'].min():.2f}  ({df.loc[df['close'].idxmin(), 'date'].date()})")
print(f"  平均收盘价:    ¥{df['close'].mean():.2f}")
print(f"  收盘价标准差:   ¥{df['close'].std():.2f}")
print(f"  上涨天数:       {df['up'].sum()} / {len(df)}  ({df['up'].mean()*100:.1f}%)")
print("=" * 55)

# 描述性统计
print()
print("收盘价描述性统计：")
df['close'].describe().round(2)


## 7. 计算移动均线

计算收盘价的 5日、20日、60日 简单移动平均线（SMA），辅助判断趋势。


In [ ]:
# ========================================
# 计算移动均线
# ========================================
df['MA5']  = df['close'].rolling(window=5).mean()
df['MA20'] = df['close'].rolling(window=20).mean()
df['MA60'] = df['close'].rolling(window=60).mean()

print("均线计算完成 ✅")
print()
print("含均线的后 10 行：")
df[['date', 'close', 'MA5', 'MA20', 'MA60']].tail(10).round(2)


## 8. 绘制收盘价走势图

收盘价曲线 + MA5/MA20/MA60 均线，带面积填充。


In [ ]:
# ========================================
# 收盘价走势图
# ========================================
fig, ax = plt.subplots(figsize=(18, 7))
fig.patch.set_facecolor('#f8f9fb')
ax.set_facecolor('#f8f9fb')

# 收盘价面积图
ax.fill_between(df['date'], df['close'], alpha=0.08, color='#e74c3c')
ax.plot(df['date'], df['close'],
        color='#c0392b', linewidth=1.5, label='收盘价', zorder=5)

# 移动均线
ax.plot(df['date'], df['MA5'],
        color='#f39c12', linewidth=1, alpha=0.85, label='MA5')
ax.plot(df['date'], df['MA20'],
        color='#2980b9', linewidth=1, alpha=0.85, label='MA20')
ax.plot(df['date'], df['MA60'],
        color='#8e44ad', linewidth=1, alpha=0.85, label='MA60')

# 标注关键点
first_date = df['date'].iloc[0]
last_date  = df['date'].iloc[-1]

ax.annotate(f'¥{first_close:.2f}',
            xy=(first_date, first_close),
            xytext=(15, -20), textcoords='offset points',
            fontsize=10, color='#666',
            arrowprops=dict(arrowstyle='->', color='#999', lw=0.8))

ax.annotate(f'¥{last_close:.2f}  ({total_return:+.1f}%)',
            xy=(last_date, last_close),
            xytext=(-15, 15), textcoords='offset points',
            fontsize=11, fontweight='bold',
            color='#e74c3c' if total_return >= 0 else '#27ae60',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=0.8))

# 坐标轴格式
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'¥{y:.2f}'))

ax.set_title('京东方A (000725.SZ) 收盘价走势 | 红涨绿跌', fontsize=16, fontweight='bold', pad=18)
ax.set_ylabel('收盘价 (¥)', fontsize=12)
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(loc='upper left', framealpha=0.9, fontsize=10)

# 标出期间最高/最低收盘价
max_c = df['close'].max()
min_c = df['close'].min()
max_date = df.loc[df['close'].idxmax(), 'date']
min_date = df.loc[df['close'].idxmin(), 'date']

ax.scatter([max_date], [max_c], color='#e74c3c', s=60, zorder=6, edgecolors='white', linewidth=1.5)
ax.scatter([min_date], [min_c], color='#27ae60', s=60, zorder=6, edgecolors='white', linewidth=1.5)
ax.annotate(f' 高 ¥{max_c:.2f}', (max_date, max_c),
            fontsize=9, color='#e74c3c', va='bottom')
ax.annotate(f' 低 ¥{min_c:.2f}', (min_date, min_c),
            fontsize=9, color='#27ae60', va='top')

plt.tight_layout()
plt.show()

print(f"最低收盘价 ¥{min_c:.2f} 出现在 {min_date.date()}")
print(f"最高收盘价 ¥{max_c:.2f} 出现在 {max_date.date()}")


## 9. 进阶：K 线图 + 成交量（上下联动）

补充完整 K 线视图，便于对比收盘价与日内波动。


In [ ]:
from matplotlib.patches import Rectangle

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10),
                                gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#f8f9fb')
for ax in (ax1, ax2):
    ax.set_facecolor('#f8f9fb')

bar_width = max(0.4, 0.7 * (len(df) / 250))
body_width = bar_width * 0.6

# --- K 线图 ---
for i, row in df.iterrows():
    date_num = mdates.date2num(row['date'])
    body_high = max(row['open'], row['close'])
    body_low  = min(row['open'], row['close'])

    if row['up']:
        color, edge = '#e74c3c', '#c0392b'
    else:
        color, edge = '#27ae60', '#1e8449'

    # 影线
    ax1.plot([date_num, date_num], [row['low'], row['high']],
             color=edge, linewidth=0.8, alpha=0.8)
    # 实体
    if abs(row['open'] - row['close']) < 0.001:
        ax1.plot([date_num - body_width/2, date_num + body_width/2],
                 [row['open'], row['close']], color=edge, linewidth=1.5)
    else:
        rect = Rectangle((date_num - body_width/2, body_low),
                         body_width, body_high - body_low,
                         facecolor=color, edgecolor=edge, linewidth=0.5, alpha=0.9)
        ax1.add_patch(rect)

# 收盘价叠加
ax1.plot(df['date'], df['close'], color='#34495e', linewidth=1.2, alpha=0.6, label='收盘价')

ax1.set_title('京东方A (000725.SZ) K线 + 收盘价 | 红涨绿跌', fontsize=14, fontweight='bold')
ax1.set_ylabel('价格 (¥)', fontsize=11)
ax1.grid(True, alpha=0.2, linestyle='--')
ax1.legend(loc='upper left', fontsize=9)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'¥{y:.2f}'))

# --- 成交量 ---
vol_colors = ['#e74c3c' if up else '#27ae60' for up in df['up']]
ax2.bar(df['date'], df['volume'], width=bar_width*0.8,
        color=vol_colors, alpha=0.5)

# 成交量均线
df['vol_ma20'] = df['volume'].rolling(20).mean()
ax2.plot(df['date'], df['vol_ma20'], color='#3498db', linewidth=1.2, alpha=0.8, label='20日均量')

ax2.set_ylabel('成交量', fontsize=11)
ax2.set_xlabel('日期', fontsize=11)
ax2.grid(True, alpha=0.2, linestyle='--')
ax2.legend(loc='upper left', fontsize=9)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y/1e8:.1f}亿'))

for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()


## 10. 总结

### 数据来源与工具
| 项目 | 说明 |
|------|------|
| 数据接口 | 腾讯自选股行情接口（日线数据） |
| 标的 | 京东方A (000725.SZ) |
| 时间范围 | 2025-07-02 ~ 2026-07-02 |
| 交易日数 | 243 天 |
| 可视化 | matplotlib |

### 关键发现
- 京东方A 一年内收盘价从 ¥3.93 涨至 ¥9.15，**累计涨幅约 +133%**
- 整体呈稳健上升趋势，MA20/MA60 均线持续向上发散，多头排列
- MA5 短期均线在上涨过程中多次回踩 MA20 后继续上行，形态健康
- 成交量在 2026 年有明显放大，与价格上涨同步，量价配合良好

### 技术要点
- 使用 `matplotlib.dates` 处理日期轴格式
- `rolling(window).mean()` 计算移动均线
- `Rectangle` 手绘 K 线蜡烛图，遵循中国股市配色（红涨绿跌）
